In [1]:
import os
import json
import numpy as np
import jax
import jax.numpy as jnp

from functools import partial
from jax import random, lax, vmap
from tqdm.notebook import tqdm

In [2]:
# initializers
def make_couplings(Lx, Ly, J=1.0, sector=(0, 0)):
    """
    sector = (tx, ty)

    tx=1:
        antiperiodic around x-cycle

    ty=1:
        antiperiodic around y-cycle
    i.e. tx = 1 should be read as 'spin flip along x-cycle: True'
    """

    tx, ty = sector

    Jx = J * jnp.ones((Ly, Lx))
    Jy = J * jnp.ones((Ly, Lx))

    if tx:
        Jx = Jx.at[:, Lx - 1].set(-J)

    if ty:
        Jy = Jy.at[Ly - 1, :].set(-J)

    return Jx, Jy

def random_spins(key, Lx, Ly):
    return random.choice(key,
                         jnp.array([-1, 1], dtype=jnp.int8),
                         shape=(Ly, Lx))


# save data in compressed form for later retrieval

def pack_spins(spins):
    """
    spins:
        {-1,+1} array

    returns:
        packed uint8 array
    """

    bits = np.asarray(spins > 0, dtype=np.uint8)

    return np.packbits(bits.reshape(-1))

In [3]:
def local_field(spins, Jx, Jy):

    left  = jnp.roll(spins,  1, axis=1)
    right = jnp.roll(spins, -1, axis=1)

    up    = jnp.roll(spins,  1, axis=0)
    down  = jnp.roll(spins, -1, axis=0)

    Jx_left = jnp.roll(Jx, 1, axis=1)
    Jy_up   = jnp.roll(Jy, 1, axis=0)

    return (Jx * right + Jx_left + Jy * down + Jy_up * up)


def energy(spins, Jx, Jy):

    sx = jnp.roll(spins, -1, axis=1)
    sy = jnp.roll(spins, -1, axis=0)

    ex = -jnp.sum(Jx * spins * sx)
    ey = -jnp.sum(Jy * spins * sy)

    return ex + ey


def magnetization(spins):
    return jnp.mean(spins)

def seam_observable(spins):
    return jnp.mean(spins[:, -1] * spins[:, 0])

In [4]:
# Metropolis updates by Checkerboard


# bipartition into white and black sites. The nearest neighbor of every white (black) site is black (white)

def checker_masks(Lx, Ly):

    yy, xx = jnp.indices((Ly, Lx))

    black = ((xx + yy) % 2 == 0)
    white = ~black

    return black, white

# metropolis update rule on black sites (white static) or white sites (black static)
def sublattice_update(spins, beta, Jx, Jy, mask, key):

    h = local_field(spins, Jx, Jy)

    dE = 2.0 * spins * h

    u = random.uniform(key, shape=spins.shape)

    accept = ((dE <= 0) | (jnp.log(u) < -beta * dE) )

    flip = mask & accept 

    spins = jnp.where(flip, -spins, spins) 

    return spins

# update on black, then update on white
def sweep(spins, beta, Jx, Jy, key, masks):

    k1, k2 = random.split(key)

    black, white = masks

    spins = sublattice_update(spins, beta, Jx, Jy, black, k1)

    spins = sublattice_update(spins, beta, Jx, Jy, white, k2)

    return spins

In [5]:
@partial(jax.jit, static_argnames=("thin",))
def evolve(spins, key, beta, Jx, Jy, masks, thin):

    def body(carry, _):

        s, k = carry

        k, kk = random.split(k)

        s = sweep(s, beta, Jx, Jy, kk, masks)

        return (s, k), None

    (spins, key), _ = lax.scan(body,(spins, key), None, length=thin)

    return spins, key

In [6]:
def run_and_stream(outdir, beta, Lx, Ly, sector=(0, 0), burn=5000,    
                   steps=15000, thin=20, seed=0):

    os.makedirs(outdir, exist_ok=True)

    tag = (f"L{Lx}"f"_beta{beta:.6f}"f"_sector{sector[0]}{sector[1]}")

    filename = os.path.join(outdir,f"{tag}.bin")

    meta_filename = os.path.join(outdir,f"{tag}.json")

    # --------------------------------------------------------
    # initialization
    # --------------------------------------------------------

    Jx, Jy = make_couplings(Lx,Ly,sector=sector)

    masks = checker_masks(Lx, Ly)

    key = random.PRNGKey(seed)

    key, k0 = random.split(key)

    spins = random_spins(k0, Lx, Ly)

    # --------------------------------------------------------
    # Burn in
    # --------------------------------------------------------

    spins, key = evolve(spins,key,beta,Jx,Jy,masks,burn)

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    metadata = {"Lx": Lx,"Ly": Ly,"beta": float(beta),"sector": sector,
                "burn": burn, "steps": steps, "thin": thin,"dtype": "packbits(uint8)"}

    with open(meta_filename, "w") as f:
        json.dump(metadata, f, indent=2)

    # --------------------------------------------------------
    # Streaming write
    # --------------------------------------------------------

    with open(filename, "wb") as f:

        for n in range(steps):

            spins, key = evolve(spins, key, beta, Jx, Jy, masks, thin)

            packed = pack_spins(spins)

            packed.tofile(f)

    #         if n % 100 == 0:
    #             print(f"step {n}/{steps}", end="\r")

    # print(f"\nFinished: {tag}")

In [7]:
# critical temperature for the Ising Model
BETA_C = 0.44068679350977147

critical_betas = np.linspace(0.438,0.443,20)

# L for Finite Size Scaling
sizes = [24, 32, 48, 56]

sectors = [(0, 0),(1, 0),(0, 1),(1, 1)]

sector_names = {(0,0):"PP", (0,1):"PA", (1,0):"AP", (1,1):"AA"}

In [8]:
pbar_sector = tqdm(sectors, desc="sectors")
    
for i_sector, sector in enumerate(pbar_sector):
    sector_label = sector_names.get(sector, str(sector))
    todo_sectors = len(sectors) - i_sector - 1

    for i_L, L in enumerate(sizes):
        todo_L = len(sizes) - i_L - 1

        pbar_sector.set_postfix(sector=sector_label, L=L,todo_L=todo_L,todo_sectors=todo_sectors,position=0)

        for beta in tqdm(critical_betas, desc=f"beta (sector={sector}, L={L})", position=1,leave=False):
            run_and_stream(outdir="../ising_mcdata", beta=beta,Lx=L,Ly=L,
                           sector=sector, burn=5000, steps=10000,thin=10,
                           seed=1234)

sectors:   0%|          | 0/4 [00:00<?, ?it/s]

beta (sector=(0, 0), L=24):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(0, 0), L=32):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(0, 0), L=48):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(0, 0), L=56):   0%|          | 0/20 [00:00<?, ?it/s]

...

beta (sector=(0, 1), L=56):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(1, 1), L=24):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(1, 1), L=32):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(1, 1), L=48):   0%|          | 0/20 [00:00<?, ?it/s]

beta (sector=(1, 1), L=56):   0%|          | 0/20 [00:00<?, ?it/s]